In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/salutsirius/enveda-casmi-2026-molecule-id-notebook02a72b3003/__results__.html
/kaggle/input/notebooks/salutsirius/enveda-casmi-2026-molecule-id-notebook02a72b3003/checkpoint.pt
/kaggle/input/notebooks/salutsirius/enveda-casmi-2026-molecule-id-notebook02a72b3003/__notebook__.ipynb
/kaggle/input/notebooks/salutsirius/enveda-casmi-2026-molecule-id-notebook02a72b3003/__output__.json
/kaggle/input/notebooks/salutsirius/enveda-casmi-2026-molecule-id-notebook02a72b3003/custom.css
/kaggle/input/notebooks/salutsirius/enveda-casmi-2026-molecule-id-notebook02a72b3003/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet


In [2]:
import torch
import torch.nn as nn
import numpy as np

class SpectrumTokenizer:
    def __init__(self, mz_min=0.0, mz_max=1800.0, mz_bin_width=0.02, max_peaks=256):
        self.mz_min = mz_min
        self.mz_max = mz_max
        self.mz_bin_width = mz_bin_width
        self.max_peaks = max_peaks
        self.mz_bins = int((mz_max - mz_min) / mz_bin_width) + 1

    def __call__(self, mzs, intensities):
        mzs = np.asarray(mzs)
        intensities = np.asarray(intensities)

        # 1. cap to top-N by intensity
        if len(mzs) > self.max_peaks:
            top = np.argsort(intensities)[-self.max_peaks:]
            mzs = mzs[top]
            intensities = intensities[top]

        # 2. sort kept peaks by m/z
        order = np.argsort(mzs)
        mzs = mzs[order]
        intensities = intensities[order]

        # 3. bin m/z with clipping
        mz_idx = np.clip(
            ((mzs - self.mz_min) / self.mz_bin_width).astype(np.int64),
            0, self.mz_bins - 1
        )

        return {
            'mz_idx': torch.tensor(mz_idx, dtype=torch.long),
            'intensity': torch.tensor(intensities, dtype=torch.float32),
            'length': len(mz_idx),
        }

In [3]:
class PeakEmbedding(nn.Module):
    def __init__(self, d_model, mz_bins):
        super().__init__()
        self.mz_emb = nn.Embedding(mz_bins, d_model)
        self.intensity_proj = nn.Linear(1, d_model)

    def forward(self, mz_idx, intensity):
        # mz_idx: [B, L], intensity: [B, L]
        mz_vec = self.mz_emb(mz_idx)                       # [B, L, d]
        # intensity.unsqueeze(-1) - [B, L] -> [B, L, 1]
        # (output = input @ weight.T + bias); [B, L, d] = [B, L, 1] @ [1, d]
        int_vec = self.intensity_proj(intensity.unsqueeze(-1))  # [B, L, d] 
        
        return mz_vec + int_vec

In [4]:
class SpectrumCollator:
    def __init__(self, max_peaks=256, pad_mz_idx=0, pad_intensity=0.0):
        self.max_peaks = max_peaks
        self.pad_mz_idx = pad_mz_idx
        self.pad_intensity = pad_intensity

    def __call__(self, batch):
        # batch: list of dicts from SpectrumTokenizer, each with
        #   'mz_idx': LongTensor [len_i]
        #   'intensity': FloatTensor [len_i]
        #   plus label fields: 'smiles', 'inchikey14', etc.

        B = len(batch)
        L = self.max_peaks

        mz_idx = torch.full((B, L), self.pad_mz_idx, dtype=torch.long)
        intensity = torch.full((B, L), self.pad_intensity, dtype=torch.float32)
        mask = torch.zeros((B, L), dtype=torch.bool)

        for i, sample in enumerate(batch):
            n = min(len(sample['mz_idx']), L)
            mz_idx[i, :n] = sample['mz_idx'][:n]
            intensity[i, :n] = sample['intensity'][:n]
            mask[i, :n] = True

        out = {
            'mz_idx': mz_idx,
            'intensity': intensity,
            'mask': mask,
        }

        # pass through label fields unchanged (as lists)
        for key in batch[0]:
            if key not in ('mz_idx', 'intensity'):
                out[key] = [sample[key] for sample in batch]

        return out

In [5]:
from torch.utils.data import Dataset

class SpectrumDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        sample = self.tokenizer(
            row['ms2_mzs'],
            row['ms2_normalized_intensities'],
        )
        sample['smiles'] = row['normalized_smiles']
        sample['inchikey14'] = row['inchikey14']
        return sample

In [6]:
import math
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, causal=False) -> None:
    super().__init__()

    self.d_k = d_k
    self.n_heads = n_heads

    self.key = nn.Linear(d_model, d_k * n_heads)
    self.query = nn.Linear(d_model, d_k * n_heads)
    self.value = nn.Linear(d_model, d_k * n_heads)

    self.fc = nn.Linear(d_k * n_heads, d_model)

    self.causal = causal
    if causal:
      cm = torch.tril(torch.ones(max_len, max_len))
      self.register_buffer(
        'causal_mask',
        cm.view(1, 1, max_len, max_len)
    )


  def forward(self, q, k, v, pad_mask=None):
    q = self.query(q)   # (N, T, d_model) -> (N, T, h*d_k)
    k = self.key(k)   # (N, T, d_model) -> (N, T, h*d_k)
    v = self.value(v)   # (N, T, d_model) -> (N, T, h*d_k)

    N = q.shape[0]
    T_output = q.shape[1]
    T_input = k.shape[1]

    # change the shape
    # (N, T, h, d_k) -> (N, h, T, d_k)
    # in order for matrix multiply to work properly
    q = q.view(N, T_output, self.n_heads, self.d_k).transpose(1, 2)
    k = k.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)
    v = v.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)

    # Compute attention weights
    # (N, h, T, d_k) x (N, h, d_k, T) -> (N, h, T, T)
    attn_scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
    if pad_mask is not None:
      attn_scores = attn_scores.masked_fill(pad_mask[:, None, None, :] == 0, float('-inf'))

    if self.causal:
      attn_scores = attn_scores.masked_fill(self.causal_mask[:, :, :T_output, :T_input] == 0, float('-inf'))

    attn_weights = F.softmax(attn_scores, dim=-1)

    # Compute attention weights values
    # (N, h, T, T) x (N, h, T, d_k) -> (N, h, T, d_k)
    A = attn_weights @ v

    # Reshape it back before final linear layer
    A = A.transpose(1, 2) # (N, T, h, d_k)
    A = A.contiguous().view(N, T_output, self.d_k * self.n_heads) # (N, T, h*d_k)

    # projection
    return self.fc(A)

In [7]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=256, dropout_prob=0.1) -> None:
    super().__init__()

    self.dropout = nn.Dropout(p=dropout_prob)

    position = torch.arange(max_len).unsqueeze(1)
    exp_term = torch.arange(0, d_model, 2)
    div_term = torch.exp(exp_term * (-math.log(10000.0)/d_model))
    pe = torch.zeros(1, max_len, d_model)
    pe[0, :, 0::2] = torch.sin(position * div_term)
    pe[0, :, 1::2] = torch.cos(position * div_term)
    self.register_buffer('pe', pe)

  def forward(self, x):
    # x.shape (N, T, D)
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)

In [8]:
class EncoderBlock(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, dropout_prob=0.1) -> None:
    super().__init__()

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.mha = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=False)
    self.ann = nn.Sequential(
        nn.Linear(d_model, d_model * 4),
        nn.GELU(),
        nn.Linear(d_model * 4, d_model),
        nn.Dropout(dropout_prob)
    )
    self.dropout = nn.Dropout(p=dropout_prob)

  def forward(self, x, mask=None):
    x = self.ln1(x + self.mha(x, x, x, mask))
    x = self.ln2(x + self.ann(x))
    x = self.dropout(x)
    return x

In [9]:
class Encoder(nn.Module):
  def __init__(self, mz_bins, max_len, d_k, d_model, n_heads, n_layers,
               #n_classes,
               dropout_prob) -> None:
    super().__init__()

    # self.embedding = nn.Embedding(vocab_size, d_model)
    self.embedding = PeakEmbedding(d_model, mz_bins)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)
    transformer_blocks = [
        EncoderBlock(d_k, d_model, n_heads, max_len, dropout_prob) for _ in range(n_layers)]
    self.transformer_blocks = nn.Sequential(*transformer_blocks)
    self.ln = nn.LayerNorm(d_model)
    #self.fc = nn.Linear(d_model, n_classes)

  def forward(self, mz_idx, intensity, mask=None):
    x = self.embedding(mz_idx, intensity)
    x = self.pos_encoding(x)

    for block in self.transformer_blocks:
      x = block(x, mask)

    # many-to-one x has shape (N x T x D) -> (N x D)
    #x = x[:, 0, :]

    # Masked mean pooling — averages the encoder output over real peaks only, ignoring padding
    mask_expanded = mask.unsqueeze(-1).float()  # [B, L, 1]
    x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1)
      
    x = self.ln(x)
    #x = self.fc(x)
    return x

In [10]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 110.4 MB/s  0:00:00 eta 0:00:01


In [11]:
from rdkit.Chem import rdFingerprintGenerator
import rdkit.Chem as Chem

class MorganEncoder:
    def __init__(self, radius=2, n_bits=2048):
        self.gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

    def __call__(self, smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        fp = self.gen.GetFingerprintAsNumPy(mol)   # returns np.uint8 array directly
        return fp.astype(np.float32)

In [12]:
class ProjectionHead(nn.Module):
    def __init__(self, d_model=256, d_fp=2048):
        super().__init__()
        self.proj = nn.Linear(d_model, d_fp)

    def forward(self, x):
        return self.proj(x)

In [13]:
from pathlib import Path

matches = list(Path("/kaggle/input").rglob("checkpoint.pt"))
path = matches[0].parent
checkpoint_path = list(Path("/kaggle/input").rglob("checkpoint.pt"))[0]


In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = SpectrumTokenizer()
enc = Encoder(
    mz_bins=tok.mz_bins,      # 90001
    max_len=256,
    d_k=64,
    d_model=256,
    n_heads=4,
    n_layers=4,
    dropout_prob=0.1).to(device)

proj = ProjectionHead().to(device)

state = torch.load(checkpoint_path, map_location=device)
enc.load_state_dict(state['enc'])
proj.load_state_dict(state['proj'])
enc.eval();
proj.eval()

ProjectionHead(
  (proj): Linear(in_features=256, out_features=2048, bias=True)
)

In [15]:
train = pd.read_parquet(
    '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet',
    columns=['inchikey14', 'ms2_mzs', 'ms2_normalized_intensities', 'normalized_smiles', 'precursor_mz', 'num_peaks', 'ingest_lib']
)

In [16]:
# Split
unique_mols = train['inchikey14'].unique()
rng = np.random.default_rng(42)
val_mols = rng.choice(unique_mols, size=int(0.05 * len(unique_mols)), replace=False)
val_mask = train['inchikey14'].isin(val_mols)

train_df = train[~val_mask].reset_index(drop=True)
val_df   = train[val_mask].reset_index(drop=True)

In [17]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 136.6 MB/s  0:00:00


In [18]:
import faiss

me = MorganEncoder()

val_mols = val_df.drop_duplicates('inchikey14')[['inchikey14', 'normalized_smiles']].reset_index(drop=True)
val_fps = np.stack([me(s) for s in val_mols['normalized_smiles']]).astype('float32')
faiss.normalize_L2(val_fps)

index = faiss.IndexFlatIP(2048)
index.add(val_fps)
val_smiles_list = val_mols['normalized_smiles'].tolist()
val_ik_list = val_mols['inchikey14'].tolist()

In [19]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

collate = SpectrumCollator(max_peaks=256)

ds_val = SpectrumDataset(val_df, tok)
loader_val = DataLoader(ds_val, batch_size=64, collate_fn=collate, shuffle=False, num_workers=2)

ranks = []
with torch.no_grad():
    for batch in tqdm(loader_val, desc='validating'):
        mz = batch['mz_idx'].to(device)
        inten = batch['intensity'].to(device)
        m = batch['mask'].to(device)

        q = proj(enc(mz, inten, m))
        q = F.normalize(q, dim=-1).cpu().numpy().astype('float32')
        faiss.normalize_L2(q)

        D, I = index.search(q, k=25)

        for i, true_ik in enumerate(batch['inchikey14']):
            hits = [val_ik_list[j] for j in I[i]]
            if true_ik in hits:
                ranks.append(hits.index(true_ik) + 1)
            else:
                ranks.append(None)

# MRR
mrr = np.mean([1.0/r if r else 0.0 for r in ranks])
print('MRR@25:', mrr)
print('Hit rate@25:', np.mean([r is not None for r in ranks]))
print('Median rank among hits:', np.median([r for r in ranks if r]))

validating:   0%|          | 0/1896 [00:00<?, ?it/s]

MRR@25: 0.029745549441248378
Hit rate@25: 0.1269074451158688
Median rank among hits: 9.0
